# Epoch Monitor: Official Static 3DSSG

Use this notebook in a **separate Jupyter kernel** while the training notebook continues running.

It copies a completed epoch checkpoint before evaluating it, so it never reads a checkpoint while it is being written and never writes into the active training checkpoint folder. CPU is the default device to avoid competing for the training GPU.

`MAX_SCENES = 20` is a quick, consistent validation slice for checking the trend between epochs. Set it to `None` only after training finishes for the full official validation result.


In [1]:
from pathlib import Path
import re

ROOT = Path(r"D:\MTP_Project\MTP_Pipeline_3RScan")
CHECKPOINT_DIR = ROOT / "pipeline_outputs" / "official_static_pretrained_pointnet_run" / "checkpoints"
MONITOR_DIR = ROOT / "pipeline_outputs" / "official_static_pretrained_pointnet_monitor" / "checkpoints"
DATABASE = ROOT / "pipeline_outputs" / "3rscan_official_static_database_v2"
OFFICIAL_SPLITS = ROOT / "official_splits"

# Choose any epoch that has already completed and appeared in CHECKPOINT_DIR.
EPOCH = 16
# Keep CPU while training uses CUDA. Use None only after training has finished.
DEVICE = "cpu"
# Full official validation split. Keep None to evaluate every official validation entry.
MAX_SCENES = None

MONITOR_DIR.mkdir(parents=True, exist_ok=True)
print(f"Monitoring epoch {EPOCH} on {DEVICE}; validation scenes: {MAX_SCENES or 'all'}")


Monitoring epoch 16 on cpu; validation scenes: all


In [2]:
available = []
for path in CHECKPOINT_DIR.glob("integration_by_parts_3rscan_epoch_*.pt"):
    match = re.search(r"epoch_(\d+)$", path.stem)
    if match:
        available.append((int(match.group(1)), path.stat().st_mtime, path))

for epoch, modified, path in sorted(available):
    print(f"epoch {epoch:>2}: {path.name}")

completed_epochs = {epoch for epoch, _, _ in available}
assert EPOCH in completed_epochs, f"Epoch {EPOCH} is not complete yet. Choose one of: {sorted(completed_epochs)}"


epoch  1: integration_by_parts_3rscan_epoch_1.pt
epoch  2: integration_by_parts_3rscan_epoch_2.pt
epoch  3: integration_by_parts_3rscan_epoch_3.pt
epoch  4: integration_by_parts_3rscan_epoch_4.pt
epoch  5: integration_by_parts_3rscan_epoch_5.pt
epoch  6: integration_by_parts_3rscan_epoch_6.pt
epoch  7: integration_by_parts_3rscan_epoch_7.pt
epoch  8: integration_by_parts_3rscan_epoch_8.pt
epoch  9: integration_by_parts_3rscan_epoch_9.pt
epoch 10: integration_by_parts_3rscan_epoch_10.pt
epoch 11: integration_by_parts_3rscan_epoch_11.pt
epoch 12: integration_by_parts_3rscan_epoch_12.pt
epoch 13: integration_by_parts_3rscan_epoch_13.pt
epoch 14: integration_by_parts_3rscan_epoch_14.pt
epoch 15: integration_by_parts_3rscan_epoch_15.pt
epoch 16: integration_by_parts_3rscan_epoch_16.pt
epoch 17: integration_by_parts_3rscan_epoch_17.pt


In [3]:
import shutil

source_checkpoint = CHECKPOINT_DIR / f"integration_by_parts_3rscan_epoch_{EPOCH}.pt"
monitor_checkpoint = MONITOR_DIR / source_checkpoint.name

# Copy a stable completed checkpoint; the evaluator writes only beside this copy.
if (not monitor_checkpoint.exists()) or monitor_checkpoint.stat().st_size != source_checkpoint.stat().st_size:
    shutil.copy2(source_checkpoint, monitor_checkpoint)

print(f"Evaluating copy: {monitor_checkpoint}")


Evaluating copy: D:\MTP_Project\MTP_Pipeline_3RScan\pipeline_outputs\official_static_pretrained_pointnet_monitor\checkpoints\integration_by_parts_3rscan_epoch_16.pt


In [ ]:
import argparse
from mtp_pipeline.config import ProjectPaths
from mtp_pipeline.evaluate_3dssg import evaluate_model

args = argparse.Namespace(
    checkpoint=monitor_checkpoint,
    reference_root=ProjectPaths().reference_root,
    output_root=MONITOR_DIR.parent,
    database=DATABASE,
    mode="static",
    train_scans=OFFICIAL_SPLITS / "train_scans.txt",
    val_scans=OFFICIAL_SPLITS / "validation_scans.txt",
    split_seed=42,
    device=DEVICE,
    max_scenes=MAX_SCENES,
)
evaluate_model(args)


Protocol: OCRL-3DSSG official subset: 160 objects, 26 positive multi-label relations
Evaluating 548 official validation scenes on cpu.


Evaluating official OCRL 3DSSG protocol:   3%|█▏                                      | 17/548 [01:11<40:42,  4.60s/it]

In [ ]:
import json

result_path = monitor_checkpoint.parent / f"evaluation_official_{monitor_checkpoint.stem}.json"
results = json.loads(result_path.read_text(encoding="utf-8"))

def percent(value):
    return f"{100 * value:.2f}%"

print(f"Evaluated scenes: {results['evaluated_scenes']}")
print("\nTable 2")
for section, metrics in results["base_paper_table_2"].items():
    print(section, {name: percent(value) for name, value in metrics.items()})
print("\nSGCls without graph constraints")
print({name: percent(value) for name, value in results["base_paper_table_3"]["SGCls"]["without_graph_constraints"].items()})
print("\nSGCls mean recall without graph constraints")
print({name: percent(value) for name, value in results["base_paper_table_10_mean_recall"]["SGCls"]["without_graph_constraints"].items()})
print(f"\nSaved: {result_path}")
